In [29]:
import numpy as np
import matplotlib.pyplot as plt
import random
import math

class Value:

    # a Value represents a single scalar value and its gradient
    def __init__(self, data, _children=()):
        self.data = data
        self.grad = 0
        self._backward = lambda: None
        self._prev = set(_children)

    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward

        return out
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out
    
    def __pow__(self, other):
        # assert isinstance(other, (int, float))
        out = Value(self.data**other, (self,))

        def _backward():
            self.grad += (other * self.data**(other-1)) * out.grad
        out._backward = _backward

        return out
    
    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self, ))

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward

        return out
    
    def log(self):
        x = self.data
        out = Value(math.log(x), (self, ))

        def _backward():
            self.grad += (1 / x) * out.grad
        out._backward = _backward

        return out
    
    def relu(self):
        out = Value(0 if self.data < 0 else self.data, (self,))

        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward

        return out
    
    def sigmoid(self):
        x = self.data
        out = Value(1 / (1 + math.exp(-x)), (self, ))

        def _backward():
            s = out.data
            self.grad += (s * (1 - s)) * out.grad
        out._backward = _backward

        return out
    
    def backward(self):

        # topological order all of the children in the graph
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        # go one variable at a time and apply the chain rule to get its gradient
        self.grad = 1
        for v in reversed(topo):
            v._backward()

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"
    
    def __neg__(self): # -self
        return self * -1
    
    def __sub__(self, other): # self - other
        return self + (-other)
    
    def __rsub__(self, other): # other - self
        return other + (-self)
    
    def __radd__(self, other): # other + self
        return self + other
    
    def __rmul__(self, other): # other * self
        return self * other
    
    def __truediv__(self, other): # self / other
        return self * other**-1

    def __rtruediv__(self, other): # other / self
        return other * self**-1

In [ ]:
class Module:

    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0

    def parameters(self):
        return []

class Neuron(Module):

    def __init__(self, nin, nonlin=True):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(0)
        self.nonlin = nonlin

    def __call__(self, x):
        act = sum((wi*xi for wi,xi in zip(self.w, x)), self.b)
        return act.relu() if self.nonlin else act.sigmoid()

    def parameters(self):
        return self.w + [self.b]
    
    def __repr__(self):
        return f"{'Relu' if self.nonlin else 'Sigmoid'}Neuron({len(self.w)})"

class Layer(Module):

    def __init__(self, nin, nout, **kwargs):
        self.neurons = [Neuron(nin, **kwargs) for _ in range(nout)]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]
    
    def __repr__(self):
        return f"Layer of [{', '.join(str(n) for n in self.neurons)}]"


class MLP(Module):

    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1], nonlin=i!=len(nouts)-1) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]
    
    def __repr__(self):
        return f"MLP of [{', '.join(str(layer) for layer in self.layers)}]"

In [ ]:
X = [
    [1, 2],
    [2, 3],
    [3, 4],
    [4, 5]]

y = [0, 0, 1, 1]
model = MLP(len(X[0]), [1])

def SingleNeuronLoss(batch_size=None):
    
    # inline DataLoader :)
    if batch_size is None:
        Xb, yb = X, y
    else:
        ri = np.random.permutation(X.shape[0])[:batch_size]
        Xb, yb = X[ri], y[ri]
    inputs = [list(map(Value, xrow)) for xrow in Xb]
    
    # forward the model to get scores
    scores = list(map(model, inputs))

    losses = [
        -(yi * yhi.log() + (1 - yi) * (1 - yhi).log())
        for yi, yhi in zip(yb, scores)
    ]
    
    data_loss = sum(losses) / len(losses)

    # also get accuracy
    accuracy = [
        (scorei.data >= 0.5) == yi
        for yi, scorei in zip(yb, scores)
    ]
    print(accuracy)
    return data_loss, sum(accuracy) / len(accuracy)

total_loss, acc = SingleNeuronLoss()
print(total_loss, acc)

[True, True, True, True]
Value(data=0.5700458005188346, grad=0) 1.0


In [89]:
model = MLP(len(X[0]), [1])

In [90]:
for k in range(5):
    
    # forward
    total_loss, acc = SingleNeuronLoss()
    
    # backward
    model.zero_grad()
    total_loss.backward()
    
    # update (sgd)
    learning_rate = 1.0 - 0.9*k/100
    for p in model.parameters():
        print(p.data,p.grad)
        p.data -= learning_rate * p.grad
    
    if k % 1 == 0:
        print(f"step {k} loss {total_loss.data}, accuracy {acc*100}%")

[False, False, True, True]
-0.8212972317134066 0.27185160250477786
0.9783408482934333 0.5679099107953225
0 0.2960583082905447
step 0 loss 0.8356737759917099, accuracy 50.0%
[True, True, False, False]
-1.0931488342181845 -1.3855036470371247
0.41043093749811077 -1.6908657355006378
-0.2960583082905447 -0.30536208846351337
step 1 loss 1.3640900763059394, accuracy 50.0%
[False, False, True, True]
0.279885279995606 0.7465096353221764
2.086078881379243 1.2433488390261014
0.006555521376797047 0.4968392037039251
step 2 loss 2.8239677699851145, accuracy 50.0%
[False, False, True, True]
-0.45318718189077123 0.31731124637902364
0.8651103214556115 0.6117417122316589
-0.4813405766604574 0.2944304658526351
step 3 loss 0.735286455829099, accuracy 50.0%
[True, True, False, False]
-0.7619310246175612 -1.4199782708140447
0.2698856354542074 -1.7559139255188396
-0.7678214199350712 -0.33593565470479475
step 4 loss 1.2931066223128935, accuracy 50.0%


In [92]:
import pandas as pd
# Use pandas to read the CSV file as a dataframe
df1 = pd.read_csv("blobs600.csv")

# The y values are those labelled 'Class': extract their values
y1 = df1['Class'].values

# The x values are all other columns
del df1['Class']   # drop the 'Class' column from the dataframe
X1 = df1.values     # convert the remaining columns to a numpy array

In [93]:
# Check its dimensions

print(f"The dimensions of the dataset are: {np.shape(X1)}")

The dimensions of the dataset are: (600, 3)


In [95]:
model = MLP(len(X1[0]), [1])

def SingleNeuronLoss(batch_size=None):
    
    # inline DataLoader :)
    if batch_size is None:
        Xb, yb = X1, y1
    else:
        ri = np.random.permutation(X.shape[0])[:batch_size]
        Xb, yb = X1[ri], y1[ri]
    inputs = [list(map(Value, xrow)) for xrow in Xb]
    
    # forward the model to get scores
    scores = list(map(model, inputs))

    losses = [
        -(yi * yhi.log() + (1 - yi) * (1 - yhi).log())
        for yi, yhi in zip(yb, scores)
    ]
    
    data_loss = sum(losses) / len(losses)

    # also get accuracy
    accuracy = [
        (scorei.data >= 0.5) == yi
        for yi, scorei in zip(yb, scores)
    ]
    
    return data_loss, sum(accuracy) / len(accuracy)

for k in range(100):
    
    # forward
    total_loss, acc = SingleNeuronLoss()
    
    # backward
    model.zero_grad()
    total_loss.backward()
    
    # update (sgd)
    learning_rate = .05
    for p in model.parameters():
        p.data -= learning_rate * p.grad
    
    if k % 1 == 0:
        print(f"step {k} loss {total_loss.data}, accuracy {acc*100}%")

step 0 loss 0.7827480076509178, accuracy 52.5%
step 1 loss 0.7385360082769126, accuracy 56.49999999999999%
step 2 loss 0.6979069038439463, accuracy 59.166666666666664%
step 3 loss 0.6605750977732762, accuracy 62.16666666666667%
step 4 loss 0.6262665407315714, accuracy 65.16666666666666%
step 5 loss 0.5947217377081204, accuracy 68.83333333333333%
step 6 loss 0.5656976390366301, accuracy 71.33333333333334%
step 7 loss 0.5389686205445601, accuracy 73.66666666666667%
step 8 loss 0.5143267526908365, accuracy 75.66666666666667%
step 9 loss 0.49158153877613237, accuracy 78.16666666666666%
step 10 loss 0.47055927597834696, accuracy 79.16666666666666%
step 11 loss 0.4511021650856367, accuracy 81.66666666666667%
step 12 loss 0.43306726830795617, accuracy 83.5%
step 13 loss 0.4163253909941754, accuracy 85.83333333333333%
step 14 loss 0.40075994311192675, accuracy 87.16666666666667%
step 15 loss 0.38626582004299465, accuracy 87.5%
step 16 loss 0.3727483293584432, accuracy 88.66666666666667%
step 1